
# Comparative Analysis of Gaussian Naïve Bayes and Hybrid Models for Enhanced Classification



## 1. Imports and configuration

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, KBinsDiscretizer
from sklearn.naive_bayes import GaussianNB, CategoricalNB, MultinomialNB
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TEST_SIZE = 0.20


In [ ]:

DATA_PATH = "ObesityDataSet_raw_and_data_sinthetic.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place the obesity CSV in the same folder as this notebook "
        "or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
display(df.info())


## 3. Dataset description

In [ ]:

print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isnull().sum())

print("\nTarget distribution:")
display(df["NObeyesdad"].value_counts())


### Feature types used in the report



In [ ]:

categorical_features = [
    "Gender",
    "family_history_with_overweight",
    "FAVC",
    "CAEC",
    "SMOKE",
    "SCC",
    "CALC",
    "MTRANS"
]

continuous_features = [
    "Age",
    "Height",
    "Weight",
    "FCVC",
    "NCP",
    "CH2O",
    "FAF",
    "TUE"
]

target = "NObeyesdad"

print("Categorical predictors:", len(categorical_features))
print("Continuous predictors:", len(continuous_features))
print("Target:", target)


## 4. Class distribution

In [ ]:

plt.figure(figsize=(11, 5))
sns.countplot(data=df, x=target, order=df[target].value_counts().index)
plt.xticks(rotation=45, ha="right")
plt.title("Class Distribution of the Dataset")
plt.xlabel("NObeyesdad")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 5. Histograms of all features

In [ ]:

df.hist(figsize=(16, 14), bins=20, edgecolor="black")
plt.suptitle("Histogram Plot of All Features", y=1.02)
plt.tight_layout()
plt.show()


## 6. Histograms of continuous features

In [ ]:

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, col in zip(axes.ravel(), continuous_features):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)

plt.suptitle("Histogram Plot of Continuous Features", y=1.02)
plt.tight_layout()
plt.show()


## 7. Correlation matrix



In [ ]:

# Encode categorical variables only for correlation visualization.
corr_df = df.copy()

for col in categorical_features:
    corr_df[col] = LabelEncoder().fit_transform(corr_df[col].astype(str))

corr = corr_df.drop(columns=[target], errors="ignore").corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="viridis", center=0)
plt.title("Correlation (Pearson) Between Features")
plt.tight_layout()
plt.show()


## 8. Preprocessing



In [ ]:

data = df.copy()

# Encode categorical predictors.
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# Encode target.
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(data[target])

X = data[categorical_features + continuous_features].copy()

print("Encoded target classes:")
for i, cls in enumerate(target_encoder.classes_):
    print(i, "->", cls)

print("\nMissing values after preprocessing:")
display(X.isnull().sum())


## 9. Train/test split



In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 10. Gaussian Naïve Bayes



In [ ]:

Xg_train = X_train[continuous_features]
Xg_test = X_test[continuous_features]

gnb = GaussianNB()
gnb.fit(Xg_train, y_train)

y_pred_gnb = gnb.predict(Xg_test)

print("Gaussian Naïve Bayes Accuracy:",
      accuracy_score(y_test, y_pred_gnb))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred_gnb,
    target_names=target_encoder.classes_
))


In [ ]:

cm_gnb = confusion_matrix(y_test, y_pred_gnb)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_gnb, annot=True, fmt="d", cmap="Blues",
    xticklabels=target_encoder.classes_,
    yticklabels=target_encoder.classes_
)
plt.title("Confusion Matrix - Gaussian Naïve Bayes")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()


## 11. Categorical Naïve Bayes — equal-width binning



In [ ]:

# Discretize continuous predictors into 4 equal-width bins.
equal_width = KBinsDiscretizer(
    n_bins=4,
    encode="ordinal",
    strategy="uniform"
)

Xc_train = X_train[categorical_features + continuous_features].copy()
Xc_test = X_test[categorical_features + continuous_features].copy()

Xcw_train_cont = equal_width.fit_transform(X_train[continuous_features])
Xcw_test_cont = equal_width.transform(X_test[continuous_features])

Xcw_train = np.hstack([
    X_train[categorical_features].to_numpy(),
    Xcw_train_cont
]).astype(int)

Xcw_test = np.hstack([
    X_test[categorical_features].to_numpy(),
    Xcw_test_cont
]).astype(int)

cnb_width = CategoricalNB()
cnb_width.fit(Xcw_train, y_train)

y_pred_cnb_width = cnb_width.predict(Xcw_test)

print("Categorical Naïve Bayes Accuracy (equal width):",
      accuracy_score(y_test, y_pred_cnb_width))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred_cnb_width,
    target_names=target_encoder.classes_
))


In [ ]:

cm_width = confusion_matrix(y_test, y_pred_cnb_width)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_width, annot=True, fmt="d", cmap="Blues",
    xticklabels=target_encoder.classes_,
    yticklabels=target_encoder.classes_
)
plt.title("Confusion Matrix - Categorical NB (Equal Width)")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()


## 12. Categorical Naïve Bayes — equal-frequency binning

In [ ]:

equal_frequency = KBinsDiscretizer(
    n_bins=4,
    encode="ordinal",
    strategy="quantile"
)

Xcf_train_cont = equal_frequency.fit_transform(X_train[continuous_features])
Xcf_test_cont = equal_frequency.transform(X_test[continuous_features])

Xcf_train = np.hstack([
    X_train[categorical_features].to_numpy(),
    Xcf_train_cont
]).astype(int)

Xcf_test = np.hstack([
    X_test[categorical_features].to_numpy(),
    Xcf_test_cont
]).astype(int)

cnb_frequency = CategoricalNB()
cnb_frequency.fit(Xcf_train, y_train)

y_pred_cnb_frequency = cnb_frequency.predict(Xcf_test)

print("Categorical Naïve Bayes Accuracy (equal frequency):",
      accuracy_score(y_test, y_pred_cnb_frequency))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred_cnb_frequency,
    target_names=target_encoder.classes_
))


In [ ]:

cm_frequency = confusion_matrix(y_test, y_pred_cnb_frequency)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_frequency, annot=True, fmt="d", cmap="Blues",
    xticklabels=target_encoder.classes_,
    yticklabels=target_encoder.classes_
)
plt.title("Confusion Matrix - Categorical NB (Equal Frequency)")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()


## 13. Hybrid Gaussian + Categorical Naïve Bayes



In [ ]:

# Gaussian model on continuous variables
hybrid_gaussian = GaussianNB()
hybrid_gaussian.fit(
    X_train[continuous_features],
    y_train
)

# Categorical model on categorical variables only
hybrid_categorical = CategoricalNB()
hybrid_categorical.fit(
    X_train[categorical_features].astype(int),
    y_train
)

log_prob_gaussian = hybrid_gaussian.predict_log_proba(
    X_test[continuous_features]
)

log_prob_categorical = hybrid_categorical.predict_log_proba(
    X_test[categorical_features].astype(int)
)

# Combine log-probabilities.
hybrid_log_proba = log_prob_gaussian + log_prob_categorical

y_pred_hybrid = np.argmax(hybrid_log_proba, axis=1)

hybrid_accuracy = accuracy_score(y_test, y_pred_hybrid)

print("Hybrid Gaussian + Categorical NB Accuracy:", hybrid_accuracy)
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred_hybrid,
    target_names=target_encoder.classes_
))


In [ ]:

cm_hybrid = confusion_matrix(y_test, y_pred_hybrid)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_hybrid, annot=True, fmt="d", cmap="Blues",
    xticklabels=target_encoder.classes_,
    yticklabels=target_encoder.classes_
)
plt.title("Confusion Matrix - Hybrid Gaussian + Categorical NB")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()


## 14. Model comparison

In [ ]:

results = pd.DataFrame({
    "Model": [
        "Gaussian NB",
        "Categorical NB (Equal Width)",
        "Categorical NB (Equal Frequency)",
        "Hybrid Gaussian + Categorical NB"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_gnb),
        accuracy_score(y_test, y_pred_cnb_width),
        accuracy_score(y_test, y_pred_cnb_frequency),
        accuracy_score(y_test, y_pred_hybrid)
    ]
})

display(results)

plt.figure(figsize=(10, 6))
sns.barplot(data=results, x="Model", y="Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.title("Accuracy of Different Naïve Bayes Classifiers")
plt.tight_layout()
plt.show()


## 15. ROC curve for the hybrid model

In [ ]:

# Convert labels to one-vs-rest binary form.
y_test_bin = label_binarize(y_test, classes=np.arange(len(target_encoder.classes_)))

# Normalize the combined probabilities.
hybrid_proba = np.exp(
    hybrid_log_proba - hybrid_log_proba.max(axis=1, keepdims=True)
)
hybrid_proba = hybrid_proba / hybrid_proba.sum(axis=1, keepdims=True)

fpr = {}
tpr = {}
roc_auc = {}

for i in range(len(target_encoder.classes_)):
    fpr[i], tpr[i], _ = roc_curve(
        y_test_bin[:, i],
        hybrid_proba[:, i]
    )
    roc_auc[i] = auc(fpr[i], tpr[i])

# Micro-average ROC
fpr["micro"], tpr["micro"], _ = roc_curve(
    y_test_bin.ravel(),
    hybrid_proba.ravel()
)
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(8, 6))
plt.plot(
    fpr["micro"], tpr["micro"],
    label=f"Micro-average ROC (AUC = {roc_auc['micro']:.2f})"
)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve of Hybrid Naïve Bayes Model")
plt.legend()
plt.tight_layout()
plt.show()

print("Micro-average AUC:", roc_auc["micro"])
